In [2]:
import gymnasium as gym
import tianshou as ts
ENV_NAME = 'LunarLander-v3'


In [4]:
train_envs = ts.env.DummyVectorEnv([lambda: gym.make(ENV_NAME) for _ in range(10)])
test_envs = ts.env.DummyVectorEnv([lambda: gym.make(ENV_NAME) for _ in range(100)])


In [10]:
import numpy as np

import torch
from torch import nn

class Net(nn.Module):

    def __init__(self, state_shape, action_shape):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(np.prod(state_shape), 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, np.prod(action_shape))
        )
    
    def forward(self, obs, state=None, info={}):
        if not isinstance(obs, torch.Tensor):
            ## need device?
            obs = torch.tensor(obs, dtype=torch.float)
        batch = obs.shape[0]
        logits = self.model(obs.view(batch, -1))
        ## state could be next hidden state for a RNN
        ## first output may vary on algo type (e.g. mu, sigma or logits)
        return logits, state

env = gym.make(ENV_NAME)
state_shape = env.observation_space.shape or env.observation_space.n
action_shape = env.action_space.shape or env.action_space.n

net = Net(state_shape, action_shape)
optim = torch.optim.Adam(net.parameters(), lr = 1e-3)

## define the DQN policy
policy = ts.policy.DQNPolicy(
    model = net, # our network
    optim=optim,
    action_space=env.action_space,
    discount_factor = 0.9,
    estimation_step=1,
    target_update_freq=320
)

In [12]:
## setup collector to help policy interact with different types of environments
train_collector = ts.data.Collector(
    policy=policy, ## the dqn
    env=train_envs,
    buffer=ts.data.VectorReplayBuffer(total_size=20000, buffer_num=10), # total_size is num obs, buffer_num is number per env
    exploration_noise=True
)
test_collector = ts.data.Collector(policy, test_envs, exploration_noise=True)

In [ ]:
from torch.utils.tensorboard import SummaryWriter
from tianshou.utils import TensorboardLogger

writer=SummaryWriter('log/dqn')
## train the policy - use OffpolicyTrainer for DQN
result = ts.trainer.OffpolicyTrainer(
    policy=policy,
    train_collector=train_collector,
    test_collector=test_collector,
    max_epoch=10, step_per_epoch=10000, step_per_collect=10,
    update_per_step=0.1, episode_per_test=100, batch_size=64,
    train_fn=lambda epoch, env_step: policy.set_eps(0.1), ## could adjust this to increment epsilon
    test_fn=lambda epoch, env_step: policy.set_eps(0.05),
    stop_fn=lambda mean_rewards: mean_rewards >= 200,
    logger=TensorboardLogger(writer)
).run()


Epoch #1: 10001it [00:27, 366.72it/s, env_step=10000, gradient_step=1000, len=1540, n/ep=6, n/st=10, rew=-139.49]                           


Epoch #1: test_reward: -102.385203 ± 35.755341, best_reward: -102.385203 ± 35.755341 in #1


Epoch #2: 10001it [00:13, 747.75it/s, env_step=20000, gradient_step=2000, len=1000, n/ep=5, n/st=10, rew=-91.36]                           


Epoch #2: test_reward: -103.102927 ± 24.472048, best_reward: -102.385203 ± 35.755341 in #1


Epoch #3: 10001it [00:14, 671.54it/s, env_step=30000, gradient_step=3000, len=1000, n/ep=3, n/st=10, rew=-103.33]                           


...

Epoch #8: test_reward: -104.218663 ± 47.971173, best_reward: -102.385203 ± 35.755341 in #1


Epoch #9: 10001it [00:14, 668.94it/s, env_step=90000, gradient_step=9000, len=1000, n/ep=0, n/st=10, rew=-51.99]                           


Epoch #9: test_reward: -110.606035 ± 26.647946, best_reward: -102.385203 ± 35.755341 in #1


Epoch #10: 10001it [00:12, 780.96it/s, env_step=100000, gradient_step=10000, len=1000, n/ep=0, n/st=10, rew=-66.84]                           


Epoch #10: test_reward: -80.676578 ± 25.015165, best_reward: -80.676578 ± 25.015165 in #10


In [25]:
collect_result = train_collector.collect(n_step=5000)

In [33]:
collect_result.pprint_asdict()

CollectStats
----------------------------------------
{   'collect_speed': 3533.8726718488547,
    'collect_time': 1.4148783683776855,
    'lens': array([1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000]),
    'lens_stat': {'max': 1000.0, 'mean': 1000.0, 'min': 1000.0, 'std': 0.0},
    'n_collected_episodes': 9,
    'n_collected_steps': 5000,
    'returns': array([-125.91749271,  -17.13752445, -116.43168622,  -98.46604116,
       -113.44568977, -117.61594266,  -71.54676752,  -96.90127912,
        -81.46635827]),
    'returns_stat': {   'max': -17.137524452987538,
                        'mean': -93.21430909902301,
                        'min': -125.91749271376835,
                        'std': 31.709868091567085}}


In [34]:
import pickle

import numpy as np
import torch

from tianshou.data import Batch

In [35]:
data = Batch(a=4, b=[5,5], c="123", d = ("a", -2, -3))
print(data)
print(data.b)

Batch(
    a: array(4),
    b: array([5, 5]),
    c: '123',
    d: array(['a', '-2', '-3'], dtype=object),
)
[5 5]


In [36]:
import gymnasium as gym
import torch

from tianshou.data import Collector, VectorReplayBuffer
from tianshou.env import DummyVectorEnv
from tianshou.policy import PGPolicy
from tianshou.utils.net.common import Net
from tianshou.utils.net.discrete import Actor

In [38]:
env = gym.make("CartPole-v1")
test_envs = DummyVectorEnv(
    [lambda: gym.make("CartPole-v1") for _ in range(2)]
)
net = Net(env.observation_space.shape, hidden_sizes=[16,])

actor = Actor(net, env.action_space.n)
optim = torch.optim.Adam(actor.parameters(), lr=3.0e-3)
policy = PGPolicy(
    actor=actor,
    optim=optim,
    dist_fn=torch.distributions.Categorical,
    action_space=env.action_space,
    action_scaling=False,
)
test_collector=Collector(policy, test_envs)

In [39]:
collect_result = test_collector.collect(reset_before_collect=True, n_episode=9)

collect_result.pprint_asdict()

CollectStats
----------------------------------------
{   'collect_speed': 1018.4666301830583,
    'collect_time': 0.1973555088043213,
    'lens': array([10, 14, 13, 26, 11, 20, 41, 44, 22]),
    'lens_stat': {   'max': 44.0,
                     'mean': 22.333333333333332,
                     'min': 10.0,
                     'std': 11.897712198383164},
    'n_collected_episodes': 9,
    'n_collected_steps': 201,
    'returns': array([10., 14., 13., 26., 11., 20., 41., 44., 22.]),
    'returns_stat': {   'max': 44.0,
                        'mean': 22.333333333333332,
                        'min': 10.0,
                        'std': 11.897712198383164}}


In [40]:
train_env_num = 4
buffer_size = 100
train_envs = DummyVectorEnv([lambda: gym.make('CartPole-v1') for _ in range(train_env_num)])
replayBuffer = VectorReplayBuffer(buffer_size, train_env_num)

train_collector = Collector(policy, train_envs, replayBuffer)

In [49]:
train_envs.reset()
train_envs

In [50]:
train_envs.step(np.array([1,1,1,1]))

(array([[-0.03863444,  0.22697279, -0.03745413, -0.29182974],
        [-0.04373907,  0.23372693, -0.00975515, -0.2975421 ],
        [-0.04573995,  0.16802524, -0.02218059, -0.34518412],
        [-0.04486895,  0.21705343, -0.01645747, -0.31477118]],
       dtype=float32),
 array([1., 1., 1., 1.]),
 array([False, False, False, False]),
 array([False, False, False, False]),
 array([{'env_id': 0}, {'env_id': 1}, {'env_id': 2}, {'env_id': 3}],
       dtype=object))

In [41]:
train_collector.reset()
replayBuffer.reset()

print(f"Replay buffer before collecting is empty, and has length={len(replayBuffer)} \n")
n_step = 50
collect_result = train_collector.collect(n_step=n_step)
print(
    f"Replay buffer after collecting {n_step} steps has length={len(replayBuffer)}.\n"
    f"This may exceed n_step when it is not a multiple of train_env_num because of vectorization.\n",
)
collect_result.pprint_asdict()

Replay buffer before collecting is empty, and has length=0 

Replay buffer after collecting 50 steps has length=52.
This may exceed n_step when it is not a multiple of train_env_num because of vectorization.

CollectStats
----------------------------------------
{   'collect_speed': 2423.725738162179,
    'collect_time': 0.021454572677612305,
    'lens': array([12, 13]),
    'lens_stat': {'max': 13.0, 'mean': 12.5, 'min': 12.0, 'std': 0.5},
    'n_collected_episodes': 2,
    'n_collected_steps': 52,
    'returns': array([12., 13.]),
    'returns_stat': {'max': 13.0, 'mean': 12.5, 'min': 12.0, 'std': 0.5}}


c:\Users\61417\Documents\Research\tianshou_ewc\venv\Lib\site-packages\tianshou\data\collector.py:323: UserWarning: n_step=50 is not a multiple of (self.env_num=4), which may cause extra transitions being collected into the buffer.
  warnings.warn(


In [53]:
samp = replayBuffer.sample(10)
samp[0]['info']['env_id']

array([0, 0, 0, 0, 0, 0, 1, 2, 3, 3])

In [60]:
from typing import Tuple, List, Dict, Any
## step 1.
## create a continual env - switches to a new env after k steps
## returns a task_id (in info?)

class ContinualEnv(gym.Env):
    def __init__(self, envs: List[gym.Env], steps_per_env):
        self.action_space = envs[0].action_space
        self.observation_space = envs[0].observation_space

        self.envs = envs
        self.num_envs = len(envs)
        self.steps_per_env = steps_per_env
        self.steps_limit = self.num_envs * self.steps_per_env
        self.cur_step = 0
        self.cur_seq_idx = 0

    def step(self, action: Any) -> Tuple[np.ndarray, float, bool, bool, Dict]:
        obs, reward, terminated, truncated, info = self.envs[self.cur_seq_idx].step(action)
        info["task_id"] = self.cur_seq_idx

        self.cur_step += 1
        if self.cur_step % self.steps_per_env == 0:
            truncated = True
            info["TimeLimit.truncated"] = True

            self.cur_seq_idx += 1

        return obs, reward, terminated, truncated, info
    
    def reset(self) -> np.ndarray:
        return self.envs[self.cur_seq_idx].reset()


        

In [75]:
collector.buffer.sample(10)[0]['info']

Batch(
    env_id: array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
    task_id: array([1, 0, 0, 0, 1, 0, 0, 1, 0, 0]),
    TimeLimit.truncated: array([False, False, False, False, False, False, False, False, False,
                                False]),
)

In [78]:
results.returns_stat.mean

-256.29263864310644

In [ ]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter('log/custom/run6')

N_EPOCHS = 1000
STEP_PER_EPOCH = 10000
TASKS = [0, 1]
NUM_ENVS = 10#len(TASKS)
PARAMS = {
    0: {'id': 'LunarLander-v3', 'enable_wind':False},
    1: {'id': 'LunarLander-v3', 'enable_wind': True}
}
EVAL_EPISODES = 10
UPDATE_PER_STEP=1

_env = gym.make('LunarLander-v3')
state_shape = _env.observation_space.shape or _env.observation_space.n
action_shape = _env.action_space.shape or _env.action_space.n

net = Net(state_shape, action_shape)
optim = torch.optim.Adam(net.parameters(), lr = 1e-3)

## define the DQN policy
policy = ts.policy.DQNPolicy(
    model = net, # our network
    optim=optim,
    action_space=_env.action_space,
    discount_factor = 0.9,
    estimation_step=1,
    target_update_freq=320
)


## for each task
for task in TASKS:

    ## create an env
    env = ts.env.DummyVectorEnv([lambda: gym.make(**PARAMS[task]) for _ in range(NUM_ENVS)])
    collector = ts.data.Collector(
        policy=policy,
        env=env,
        buffer = ts.data.VectorReplayBuffer(1e5, NUM_ENVS)
    )

    ## train for N_EPOCHS = NUM_ENVS * STEPS_PER_EPOCH
    for epoch in range(N_EPOCHS):
        steps_so_far = epoch * (STEP_PER_EPOCH) * (task + 1)
        results = collector.collect(
            n_step=STEP_PER_EPOCH, 
            reset_before_collect=True
        )

        stats = results.returns_stat

        writer.add_scalar(f"train/{task}_mean", stats.mean, steps_so_far)
        writer.add_scalar(f"train/{task}_std", stats.std, steps_so_far)

        ## can update policy like this...
        policy.is_within_training_step = True
        num_updates = int(UPDATE_PER_STEP * results.n_collected_steps)
        loss=0
        for _ in range(num_updates):
            update_results = policy.update(64, collector.buffer)
            loss += update_results.loss
        policy.is_within_training_step = False

        writer.add_scalar(f"train/loss", loss / num_updates, steps_so_far)

        ## Eval one of each env
        for eval_task in TASKS:
            test_collector = ts.data.Collector(
                policy=policy,
                env = ts.env.DummyVectorEnv(
                    [lambda: gym.make(**PARAMS[eval_task]) for _ in range(EVAL_EPISODES)]
                )
            )
            eval_results = test_collector.collect(n_episode=EVAL_EPISODES, reset_before_collect=True)
            eval_stats = eval_results.returns_stat
            writer.add_scalar(f"test/{eval_task}_mean", eval_stats.mean, steps_so_far)
            writer.add_scalar(f"test/{eval_task}_std", eval_stats.std, steps_so_far)
    # print(results)
    

In [119]:
num_updates

10000

In [99]:
res = collector.collect(n_step=1)
result["n/st"]

c:\Users\61417\Documents\Research\tianshou_ewc\venv\Lib\site-packages\tianshou\data\collector.py:323: UserWarning: n_step=1 is not a multiple of (self.env_num=2), which may cause extra transitions being collected into the buffer.
  warnings.warn(


TypeError: 'InfoStats' object is not subscriptable

In [103]:
res.n_collected_steps


2

In [ ]:
from torch.utils.tensorboard import SummaryWriter
from tianshou.utils import TensorboardLogger
_env = gym.make('LunarLander-v3')
state_shape = _env.observation_space.shape or _env.observation_space.n
action_shape = _env.action_space.shape or _env.action_space.n

net = Net(state_shape, action_shape)
optim = torch.optim.Adam(net.parameters(), lr = 1e-3)

## define the DQN policy
policy = ts.policy.DQNPolicy(
    model = net, # our network
    optim=optim,
    action_space=_env.action_space,
    discount_factor = 0.9,
    estimation_step=1,
    target_update_freq=320
)

## setup collector to help policy interact with different types of environments
train_collector = ts.data.Collector(
    policy=policy, ## the dqn
    env=env,
    buffer=ts.data.VectorReplayBuffer(total_size=20000, buffer_num=1), # total_size is num obs, buffer_num is number per env
    exploration_noise=True
)
test_envs = 
test_collector = ts.data.Collector(policy, env, exploration_noise=True)

writer=SummaryWriter('log/dqn')
## train the policy - use OffpolicyTrainer for DQN
result = ts.trainer.OffpolicyTrainer(
    policy=policy,
    train_collector=train_collector,
    test_collector=test_collector,
    max_epoch=10, step_per_epoch=2000, step_per_collect=10,
    update_per_step=0.1, episode_per_test=10, batch_size=64,
    train_fn=lambda epoch, env_step: policy.set_eps(0.1), ## could adjust this to increment epsilon
    test_fn=lambda epoch, env_step: policy.set_eps(0.05),
    stop_fn=lambda mean_rewards: mean_rewards >= 200,
    logger=TensorboardLogger(writer)
).run()


Epoch #1: 2001it [00:03, 605.15it/s, env_step=2000, gradient_step=200, len=52, n/ep=0, n/st=10, rew=-152.63]                           


IndexError: list index out of range